# Patient Portal v2 Overnight Pipeline

**End-to-end v2 pipeline for 400 patients × 2000 questions on Colab GPU**

Runs the complete workflow:
1. Build FAISS index for 400 patients
2. Run Modes 1/2/3 on all 2000 questions (1600 train + 400 test) with checkpointing
3. Score responses using the same rubric as v1
4. Extract 23 router features
5. Train/eval router on 1600 train, report on 400 test
6. Final eval table

**Expected runtime:** ~8.5 hours for the main LLM cell (6-8 hours on A100, ~10-12 on T4)

## Cell 1: Setup — Install, Mount Drive, Pull Ollama

In [5]:
!pip install "numpy<2" --force-reinstall --no-deps -q

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!apt-get update -qq && apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(5)
print("Ollama started")
!ollama pull gemma2
print("Gemma2 ready")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama started

Gemma2 ready


In [3]:
!pip install "numpy<2" "thinc<8.4" "spacy<3.8" --force-reinstall -q
!pip install scispacy faiss-gpu-cu12 sentence-transformers transformers tqdm scikit-learn matplotlib torch -q
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz -q
print("Installs complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 133.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 153.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.0/865.0 kB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 162.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.8/260.8 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/134.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [3]:
import os, json, pickle, re, string, time, gc, random
import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import faiss
import spacy
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

Device: cuda


In [4]:
DRIVE_ROOT = "/content/drive/MyDrive/DL Project"
print(f"DRIVE_ROOT: {DRIVE_ROOT}")

# Create output directories
os.makedirs(os.path.join(DRIVE_ROOT, "patient_index_v2"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, "patient_router_v2"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, "patient_portal_results_v2"), exist_ok=True)

DRIVE_ROOT: /content/drive/MyDrive/DL Project


## Cell 2: Load v2 Data (400 patients, 1600 train, 400 test questions)

In [5]:
# Load 400 patients
patients = []
patients_path = os.path.join(DRIVE_ROOT, "synthetic_patients", "patients_v2.jsonl")
with open(patients_path) as f:
    for line in f:
        if line.strip():
            patients.append(json.loads(line))
patient_by_id = {p["patient_id"]: p for p in patients}
print(f"Loaded {len(patients)} v2 patients")

# Load 1600 train questions
train_questions = []
train_questions_path = os.path.join(DRIVE_ROOT, "synthetic_patients", "questions_v2.jsonl")
with open(train_questions_path) as f:
    for line in f:
        if line.strip():
            train_questions.append(json.loads(line))
print(f"Loaded {len(train_questions)} train questions")

# Load 400 test questions
test_questions = []
test_questions_path = os.path.join(DRIVE_ROOT, "synthetic_patients", "test_questions_v2.jsonl")
with open(test_questions_path) as f:
    for line in f:
        if line.strip():
            test_questions.append(json.loads(line))
print(f"Loaded {len(test_questions)} test questions")

all_questions = train_questions + test_questions
print(f"\nTotal questions: {len(all_questions)}")
print(f"Train/test split: {len(train_questions)}/{len(test_questions)}")

Loaded 400 v2 patients
Loaded 1600 train questions
Loaded 400 test questions

Total questions: 2000
Train/test split: 1600/400


## Cell 3: Build Patient FAISS Index (400 patients)

In [6]:
def patient_to_chunks(patient):
    """
    Convert a patient JSON record into a list of prose chunks for RAG indexing.
    Excludes medications (those go directly to the LLM prompt).
    """
    chunks = []
    pid = patient["patient_id"]
    name = patient["name"]
    age = patient["age"]
    gender = patient["gender"].lower()

    # Chunk 1: Demographics + Active Conditions
    conditions = patient.get("active_conditions", [])
    if conditions:
        cond_parts = []
        for c in conditions:
            cond_parts.append(f"{c['condition']} (diagnosed {c['diagnosed']})")
        cond_str = "; ".join(cond_parts)
        text = (f"{name} is a {age}-year-old {gender} with the following active "
                f"medical conditions: {cond_str}.")
    else:
        text = f"{name} is a {age}-year-old {gender} with no active medical conditions documented."

    chunks.append({
        "chunk_id": f"{pid}_demographics_conditions",
        "patient_id": pid,
        "section": "demographics_and_conditions",
        "text": text,
    })

    # Chunk 2: Past Medical History (if any)
    pmh = patient.get("past_medical_history", [])
    if pmh:
        pmh_str = "; ".join(pmh)
        chunks.append({
            "chunk_id": f"{pid}_pmh",
            "patient_id": pid,
            "section": "past_medical_history",
            "text": f"{name} has the following past medical and surgical history: {pmh_str}.",
        })

    # Chunk 3: Allergies
    allergies = patient.get("allergies", [])
    if allergies:
        allerg_parts = [f"{a['substance']} ({a['reaction']})" for a in allergies]
        text = f"{name} has the following documented drug or substance allergies: " + "; ".join(allerg_parts) + "."
    else:
        text = f"{name} has no documented drug allergies."
    chunks.append({
        "chunk_id": f"{pid}_allergies",
        "patient_id": pid,
        "section": "allergies",
        "text": text,
    })

    # Chunk 4: Recent Vitals & Labs
    vitals = patient.get("recent_vitals", {})
    if vitals:
        date = vitals.get("date", "the most recent visit")
        parts = []
        for k, v in vitals.items():
            if k == "date": continue
            label = k.replace("_", " ")
            parts.append(f"{label} {v}")
        vital_str = ", ".join(parts)
        chunks.append({
            "chunk_id": f"{pid}_vitals",
            "patient_id": pid,
            "section": "recent_vitals",
            "text": f"{name}'s recent vitals and laboratory values from {date}: {vital_str}.",
        })

    # Chunk 5: Lifestyle
    lifestyle = patient.get("lifestyle", {})
    if lifestyle:
        life_parts = []
        for k, v in lifestyle.items():
            label = k.replace("_", " ")
            life_parts.append(f"{label}: {v}")
        life_str = ". ".join(life_parts)
        chunks.append({
            "chunk_id": f"{pid}_lifestyle",
            "patient_id": pid,
            "section": "lifestyle",
            "text": f"{name}'s lifestyle factors. {life_str}.",
        })

    return chunks

# Run chunker on all 400 patients
all_chunks = []
for patient in tqdm(patients, desc="Chunking patients"):
    all_chunks.extend(patient_to_chunks(patient))

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Avg chunks per patient: {len(all_chunks) / len(patients):.1f}")
from collections import Counter
section_counts = Counter(c["section"] for c in all_chunks)
print(f"\nChunks by section:")
for section, count in section_counts.most_common():
    print(f"  {section}: {count}")

Chunking patients:   0%|          | 0/400 [00:00<?, ?it/s]


Total chunks: 1633
Avg chunks per patient: 4.1

Chunks by section:
  demographics_and_conditions: 400
  allergies: 400
  recent_vitals: 400
  lifestyle: 400
  past_medical_history: 33


In [7]:
# Load MedCPT Article Encoder
article_encoder_path = os.path.join(DRIVE_ROOT, "models", "MedCPT-Article-Encoder")
print(f"Loading Article Encoder from {article_encoder_path}...")
article_tokenizer = AutoTokenizer.from_pretrained(article_encoder_path)
article_encoder = AutoModel.from_pretrained(article_encoder_path).to(DEVICE).eval()
print("Article Encoder ready")

def encode_chunks(texts, tokenizer, model, batch_size=32, max_length=256):
    """Encode chunk text -> normalized embedding vectors."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, truncation=True, padding=True,
                            max_length=max_length, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            output = model(**encoded)
            embeddings = output.last_hidden_state[:, 0, :]  # CLS
            embeddings = F.normalize(embeddings, dim=-1)
        all_embeddings.append(embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

# Encode all chunks
print("\nEncoding patient chunks...")
chunk_texts = [c["text"] for c in all_chunks]
embeddings = encode_chunks(chunk_texts, article_tokenizer, article_encoder)
print(f"Embeddings: {embeddings.shape}  (n_chunks, dim)")

# Build FAISS index
dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)
print(f"FAISS index built: {faiss_index.ntotal} vectors, dim {dim}")

# Save index + metadata
index_dir = os.path.join(DRIVE_ROOT, "patient_index_v2")
os.makedirs(index_dir, exist_ok=True)

faiss_path = os.path.join(index_dir, "patient_index.bin")
faiss.write_index(faiss_index, faiss_path)
print(f"Saved FAISS index: {faiss_path}")

chunks_path = os.path.join(index_dir, "patient_chunks.jsonl")
with open(chunks_path, "w") as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk) + "\n")
print(f"Saved chunk metadata: {chunks_path}")

chunk_ids_path = os.path.join(index_dir, "patient_chunk_ids.json")
chunk_ids_ordered = [c["chunk_id"] for c in all_chunks]
with open(chunk_ids_path, "w") as f:
    json.dump(chunk_ids_ordered, f)
print(f"Saved chunk ID order: {chunk_ids_path}")

Loading Article Encoder from /content/drive/MyDrive/DL Project/models/MedCPT-Article-Encoder...


Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

Article Encoder ready

Encoding patient chunks...
Embeddings: (1633, 768)  (n_chunks, dim)
FAISS index built: 1633 vectors, dim 768
Saved FAISS index: /content/drive/MyDrive/DL Project/patient_index_v2/patient_index.bin
Saved chunk metadata: /content/drive/MyDrive/DL Project/patient_index_v2/patient_chunks.jsonl
Saved chunk ID order: /content/drive/MyDrive/DL Project/patient_index_v2/patient_chunk_ids.json


## Cell 4: Load MedCPT, scispaCy, PrimeKG

In [8]:
# Load Query Encoder
query_encoder_path = os.path.join(DRIVE_ROOT, "models", "MedCPT-Query-Encoder")
query_tokenizer = AutoTokenizer.from_pretrained(query_encoder_path)
query_encoder = AutoModel.from_pretrained(query_encoder_path).to(DEVICE).eval()

# Load Cross Encoder
cross_encoder_path = os.path.join(DRIVE_ROOT, "models", "MedCPT-Cross-Encoder")
cross_tokenizer = AutoTokenizer.from_pretrained(cross_encoder_path)
cross_encoder = AutoModelForSequenceClassification.from_pretrained(cross_encoder_path).to(DEVICE).eval()

print("MedCPT encoders loaded")

# Load scispaCy
nlp = spacy.load("en_ner_bc5cdr_md")
print("scispaCy loaded")

# Load PrimeKG
primekg_path = os.path.join(DRIVE_ROOT, "primekg_index.pkl")
with open(primekg_path, "rb") as f:
    name_to_triples = pickle.load(f)
print(f"PrimeKG loaded ({len(name_to_triples):,} entities)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

MedCPT encoders loaded


/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


scispaCy loaded
PrimeKG loaded (128,550 entities)


## Cell 5: Define 3-Mode Pipeline Functions

In [9]:
def encode_query(text, max_length=64):
    encoded = query_tokenizer([text], truncation=True, padding=True,
                              max_length=max_length, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = query_encoder(**encoded)
        emb = out.last_hidden_state[:, 0, :]
        emb = F.normalize(emb, dim=-1)
    return emb.cpu().numpy()


def retrieve_patient_chunks(question, patient_id, top_k=4):
    qe = encode_query(question)
    emb = patient_embeddings_by_id[patient_id]
    chunks = patient_chunks_by_id[patient_id]
    scores = (emb @ qe.T).flatten()
    ranked = np.argsort(-scores)[:top_k]
    return [{"score": float(scores[i]), "chunk_id": chunks[i]["chunk_id"],
             "section": chunks[i]["section"], "text": chunks[i]["text"]}
            for i in ranked]


def rerank_with_cross_encoder(question, candidates, top_k=3, batch_size=8):
    if not candidates: return []
    pairs = [[question, c["text"]] for c in candidates]
    scores = []
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]
        encoded = cross_tokenizer(batch, truncation=True, padding=True,
                                  return_tensors="pt", max_length=512).to(DEVICE)
        with torch.no_grad():
            logits = cross_encoder(**encoded).logits.squeeze(dim=-1)
        if logits.dim() == 0: scores.append(float(logits))
        else: scores.extend(logits.cpu().tolist())
    for c, s in zip(candidates, scores):
        c["cross_encoder_score"] = float(s)
    return sorted(candidates, key=lambda x: x["cross_encoder_score"], reverse=True)[:top_k]


def get_kg_triples_for_drugs(drug_names, top_k_per_drug=5):
    triples, seen = [], set()
    for drug in drug_names:
        for variant in [drug.lower(), drug.lower().split()[0]]:
            if variant in name_to_triples:
                for rel, y_name, y_type in name_to_triples[variant][:top_k_per_drug]:
                    key = (variant, rel, y_name)
                    if key in seen: continue
                    seen.add(key)
                    triples.append({"subject": variant, "predicate": rel, "object": y_name,
                                    "object_type": y_type,
                                    "triple_text": f"{variant} {rel} {y_name}"})
                break
    return triples


def get_kg_triples_for_question(question, top_k=3):
    doc = nlp(question)
    triples, seen = [], set()
    for ent in doc.ents:
        e = ent.text.strip().lower()
        if e in name_to_triples:
            for rel, y_name, y_type in name_to_triples[e][:top_k]:
                key = (e, rel, y_name)
                if key in seen: continue
                seen.add(key)
                triples.append({"subject": e, "predicate": rel, "object": y_name,
                                "object_type": y_type,
                                "triple_text": f"{e} {rel} {y_name}"})
    return triples

print("Retrieval + KG helper functions ready")

Retrieval + KG helper functions ready


In [10]:
# Rebuild per-patient embeddings
print("Rebuilding per-patient embeddings...")
chunk_lookup = {}
patient_chunks_by_id = {}
patient_embeddings_by_id = {}

# Reload chunks
with open(chunks_path) as f:
    for line in f:
        c = json.loads(line)
        chunk_lookup[c["chunk_id"]] = c

# Reconstruct embeddings
all_embeddings = faiss_index.reconstruct_n(0, faiss_index.ntotal)
for chunk, emb in zip(all_chunks, all_embeddings):
    pid = chunk["patient_id"]
    patient_chunks_by_id.setdefault(pid, []).append(chunk)
    patient_embeddings_by_id.setdefault(pid, []).append(emb)
for pid in patient_embeddings_by_id:
    patient_embeddings_by_id[pid] = np.array(patient_embeddings_by_id[pid])

print(f"Per-patient embeddings ready for {len(patient_chunks_by_id)} patients")

Rebuilding per-patient embeddings...
Per-patient embeddings ready for 400 patients


In [11]:
# Prompt formatting
PREDICATE_PHRASES = {
    "indication": "is indicated for", "off-label use": "is used off-label for",
    "contraindication": "is contraindicated in", "side effect": "can cause",
    "drug-drug interaction": "interacts with", "synergistic interaction": "synergistically interacts with",
    "target": "targets", "carrier": "is carried by", "transporter": "is transported by",
    "enzyme": "is metabolized by", "phenotype present": "presents with",
    "phenotype absent": "is not associated with", "associated with": "is associated with",
    "ppi": "interacts with protein", "parent-child": "is a type of",
}


def triple_to_sentence(t):
    return f"{t['subject'].capitalize()} {PREDICATE_PHRASES.get(t['predicate'].lower(), t['predicate'])} {t['object']}."


def format_prescription_block(patient):
    meds = patient.get("medications", [])
    if not meds: return "(no active prescriptions on file)"
    lines = []
    for m in meds:
        line = f"- {m['drug']} {m['dosage']}, {m['frequency']}, for {m['indication']}"
        if m.get("notes"): line += f" (Note: {m['notes']})"
        lines.append(line)
    return "\n".join(lines)


def format_patient_context(reranked):
    if not reranked: return ""
    return "=== PATIENT MEDICAL RECORD ===\n" + "\n".join(f"- {c['text']}" for c in reranked)


def format_kg_block(kg_triples):
    if not kg_triples: return ""
    return ("=== DRUG SAFETY INFORMATION (from validated drug databases) ===\n"
            + "\n".join(f"- {triple_to_sentence(t)}" for t in kg_triples))


def build_prompt_v1(question, patient, patient_context="", kg_block=""):
    """Mode 1 / Mode 2 — original prompt."""
    rx = format_prescription_block(patient)
    parts = [
        "You are a helpful medical assistant answering questions for a patient about their",
        "prescriptions and health. Use the information provided to give a clear, accurate,",
        "and patient-friendly answer. If you do not have the information needed, say so",
        "honestly rather than guessing.",
        "",
        "=== ACTIVE PRESCRIPTIONS ===", rx, "",
    ]
    if patient_context: parts += [patient_context, ""]
    if kg_block: parts += [kg_block, ""]
    parts += ["=== PATIENT QUESTION ===", question, "", "Provide a concise, helpful answer:"]
    return "\n".join(parts)


def build_prompt_v2_mode3(question, patient, patient_context, kg_block):
    """Mode 3 — diplomatic but directive prompt (the fix)."""
    rx = format_prescription_block(patient)
    return (
        "You are a helpful medical assistant answering questions for a patient about their\n"
        "prescriptions and health. Use the patient's prescriptions, medical record, and the\n"
        "drug safety information below to give a thoughtful, helpful answer.\n"
        "\n"
        "If the safety information directly addresses the question, share it in plain language.\n"
        "Recommending the patient confirm with their doctor is appropriate, but try to be\n"
        "informative first rather than only deferring. When the medical facts above clearly\n"
        "answer the question, lead with that information.\n"
        "\n"
        f"=== ACTIVE PRESCRIPTIONS ===\n{rx}\n"
        f"\n{patient_context}\n"
        f"\n{kg_block}\n"
        "\n=== PATIENT QUESTION ===\n"
        f"{question}\n"
        "\nProvide a concise, helpful answer:"
    )

print("Prompt formatters ready")

Prompt formatters ready


In [12]:
OLLAMA_URL = "http://localhost:11434/api/generate"
GEMMA2_MODEL = "gemma2"


def query_gemma2(prompt, temperature=0.0, max_tokens=512, seed=42):
    payload = {"model": GEMMA2_MODEL, "prompt": prompt, "stream": False,
               "options": {"temperature": temperature, "num_predict": max_tokens, "seed": seed}}
    try:
        resp = requests.post(OLLAMA_URL, json=payload, timeout=180)
        resp.raise_for_status()
        return resp.json().get("response", "").strip()
    except Exception as e:
        print(f"Ollama error: {e}")
        return ""


def mode_1(question, patient):
    t0 = time.time()
    prompt = build_prompt_v1(question, patient)
    answer = query_gemma2(prompt)
    return {"mode": 1, "mode_name": "LLM_only", "answer": answer,
            "latency_seconds": round(time.time() - t0, 2),
            "n_retrieved_chunks": 0, "n_kg_triples": 0,
            "retrieved_chunks": [], "kg_triples": []}


def mode_2(question, patient):
    t0 = time.time()
    candidates = retrieve_patient_chunks(question, patient["patient_id"], top_k=4)
    reranked = rerank_with_cross_encoder(question, candidates, top_k=3)
    ctx = format_patient_context(reranked)
    prompt = build_prompt_v1(question, patient, patient_context=ctx)
    answer = query_gemma2(prompt)
    return {"mode": 2, "mode_name": "RAG", "answer": answer,
            "latency_seconds": round(time.time() - t0, 2),
            "n_retrieved_chunks": len(reranked), "n_kg_triples": 0,
            "retrieved_chunks": [{"section": c["section"],
                                  "score": c.get("cross_encoder_score", c["score"]),
                                  "text": c["text"][:200]} for c in reranked],
            "kg_triples": []}


def mode_3(question, patient):
    t0 = time.time()
    candidates = retrieve_patient_chunks(question, patient["patient_id"], top_k=4)
    reranked = rerank_with_cross_encoder(question, candidates, top_k=3)
    ctx = format_patient_context(reranked)

    drug_names = [m["drug"] for m in patient.get("medications", [])]
    drug_triples = get_kg_triples_for_drugs(drug_names, top_k_per_drug=4)
    question_triples = get_kg_triples_for_question(question, top_k=3)
    seen, all_triples = set(), []
    for t in drug_triples + question_triples:
        key = (t["subject"], t["predicate"], t["object"])
        if key in seen: continue
        seen.add(key)
        all_triples.append(t)

    kg_block = format_kg_block(all_triples)
    prompt = build_prompt_v2_mode3(question, patient, ctx, kg_block)
    answer = query_gemma2(prompt)
    return {"mode": 3, "mode_name": "RAG_KG", "answer": answer,
            "latency_seconds": round(time.time() - t0, 2),
            "n_retrieved_chunks": len(reranked), "n_kg_triples": len(all_triples),
            "retrieved_chunks": [{"section": c["section"],
                                  "score": c.get("cross_encoder_score", c["score"]),
                                  "text": c["text"][:200]} for c in reranked],
            "kg_triples": [t["triple_text"] for t in all_triples]}

print("All 3 modes ready")

All 3 modes ready


## Cell 6: Run All 2000 Questions (1600 train + 400 test) WITH CHECKPOINTING — ~8 HOURS

In [13]:
responses_path = os.path.join(DRIVE_ROOT, "synthetic_patients", "patient_portal_responses_v2_2000.jsonl")

# Resume support: load already-done question_ids
done_ids = set()
if os.path.exists(responses_path):
    with open(responses_path) as f:
        for line in f:
            try:
                done_ids.add(json.loads(line)["question_id"])
            except Exception:
                pass
    print(f"Resuming: {len(done_ids)} questions already processed")
else:
    print("Starting fresh")

remaining = [q for q in all_questions if q["question_id"] not in done_ids]
print(f"To process: {len(remaining)} questions")
print(f"Expected runtime: ~{len(remaining) * 15 / 3600:.1f} hours (5s per mode × 3 modes + overhead)")

Starting fresh
To process: 2000 questions
Expected runtime: ~8.3 hours (5s per mode × 3 modes + overhead)


In [14]:
CHECKPOINT_EVERY = 50

with open(responses_path, "a") as fout:
    for i, q in enumerate(tqdm(remaining, desc="Pipelines (2000 questions, ~8h)")):
        try:
            p = patient_by_id[q["patient_id"]]
            r1 = mode_1(q["question"], p)
            r2 = mode_2(q["question"], p)
            r3 = mode_3(q["question"], p)
            record = {
                "question_id": q["question_id"],
                "patient_id": q["patient_id"],
                "question": q["question"],
                "category": q["category"],
                "expected_mode": q["expected_mode"],
                "expected_mode_name": q["expected_mode_name"],
                "rationale": q["rationale"],
                "mode_1": r1,
                "mode_2": r2,
                "mode_3": r3,
            }
            fout.write(json.dumps(record) + "\n")
            # Checkpoint every CHECKPOINT_EVERY questions
            if (i + 1) % CHECKPOINT_EVERY == 0:
                fout.flush()
                print(f"\n[Checkpoint {(i + 1) // CHECKPOINT_EVERY}] Processed {i + 1}/{len(remaining)} questions")
        except Exception as e:
            print(f"Error on {q['question_id']}: {e}")

print(f"\nDone. Saved to {responses_path}")

Pipelines (2000 questions, ~8h):   0%|          | 0/2000 [00:00<?, ?it/s]


[Checkpoint 1] Processed 50/2000 questions

[Checkpoint 2] Processed 100/2000 questions

[Checkpoint 3] Processed 150/2000 questions

[Checkpoint 4] Processed 200/2000 questions

[Checkpoint 5] Processed 250/2000 questions

[Checkpoint 6] Processed 300/2000 questions

[Checkpoint 7] Processed 350/2000 questions

[Checkpoint 8] Processed 400/2000 questions

[Checkpoint 9] Processed 450/2000 questions

[Checkpoint 10] Processed 500/2000 questions

[Checkpoint 11] Processed 550/2000 questions

[Checkpoint 12] Processed 600/2000 questions

[Checkpoint 13] Processed 650/2000 questions

[Checkpoint 14] Processed 700/2000 questions

[Checkpoint 15] Processed 750/2000 questions

[Checkpoint 16] Processed 800/2000 questions

[Checkpoint 17] Processed 850/2000 questions

[Checkpoint 18] Processed 900/2000 questions

[Checkpoint 19] Processed 950/2000 questions

[Checkpoint 20] Processed 1000/2000 questions

[Checkpoint 21] Processed 1050/2000 questions

[Checkpoint 22] Processed 1100/2000 quest

## Cell 7: Score All Responses Using Rubric from Mode_3_Prompt_Fix.ipynb

In [15]:
# Scoring helpers (copied verbatim from Mode_3_Prompt_Fix.ipynb)
PUNT_RE = re.compile(
    r"i can'?t give (you )?medical advice|i'?m sorry,? but i can'?t|"
    r"i am not (a doctor|a medical|able to)|please (talk to|consult|speak to|discuss with) (your )?doctor|"
    r"only your doctor can",
    re.IGNORECASE)


def is_pure_punt(answer):
    if not answer: return True
    if len(answer) < 250 and PUNT_RE.search(answer):
        sentences = re.split(r"[.!?]+", answer)
        useful = 0
        for s in sentences:
            s = s.strip()
            if len(s) < 15 or PUNT_RE.search(s): continue
            if any(g in s.lower() for g in ["can advise", "personalized guidance",
                                             "based on your medical", "they know your"]):
                continue
            useful += 1
        if useful <= 1: return True
    return False


def cites_patient_specific_value(answer, patient):
    a = answer.lower()
    for k, v in patient.get("recent_vitals", {}).items():
        if k == "date": continue
        v_str = str(v).lower()
        if v_str and v_str in a: return True
        if isinstance(v, (int, float)):
            if str(v) in answer or f"{v:.1f}" in answer: return True
    for al in patient.get("allergies", []):
        if al.get("substance", "").lower() in a: return True
    for pmh in patient.get("past_medical_history", []):
        first = pmh.lower().split()[:3]
        if first and " ".join(first) in a: return True
    return False


def mentions_drug_interaction(answer):
    a = answer.lower()
    return any(kw in a for kw in [
        "interact", "interaction", "increase the levels", "decrease the levels",
        "blood thinner", "bleeding risk", "cyp", "metabolism", "metabolized",
        "qt prolongation", "serotonin syndrome", "lactic acidosis",
        "hypoglycemi", "hyperkalemi", "potassium",
        "potentiate", "additive effect", "reduce effectiveness",
        "could interact", "may interact", "can interact", "interfere",
    ])


def gives_specific_safety_advice(answer):
    a = answer.lower()
    for pat in [r"avoid", r"don'?t (take|drink|consume|combine)", r"do not (take|combine)",
                r"safe to take", r"can take .* with", r"should not", r"contraindicated",
                r"limit (your )?intake", r"caution", r"may cause", r"is generally safe",
                r"best to wait", r"separate .* by", r"hours apart",
                r"could interact", r"may interact", r"can interact"]:
        if re.search(pat, a): return True
    return False


def is_informative_diplomatic(answer, patient):
    if len(answer) < 200: return False
    a = answer.lower()
    has_specific = mentions_drug_interaction(answer)
    if not has_specific:
        for med in patient.get("medications", []):
            fw = med["drug"].split()[0].lower()
            if fw in a and len(fw) > 3:
                has_specific = True; break
    if not has_specific: return False
    sentences = re.split(r"[.!?]+", answer)
    sub = 0
    for s in sentences:
        sl = s.strip().lower()
        if len(sl) < 25 or PUNT_RE.search(sl): continue
        if any(g in sl for g in ["personalized guidance", "based on your medical history",
                                  "they know your", "let me know if you"]): continue
        sub += 1
    return sub >= 2


def score_mode(record_field, patient, expected, rationale):
    answer = record_field["answer"]
    if not answer or len(answer) < 30: return 0, "empty/too short"
    if is_pure_punt(answer): return 0, "punted"
    mode_idx = record_field["mode"]

    if expected == "LLM_only":
        return 1, "generic question, all modes handle it"

    if expected == "RAG":
        if mode_idx == 1:
            if any(k in rationale.lower() for k in ["notes", "prescription", "indication"]):
                if any(m["drug"].lower() in answer.lower() for m in patient.get("medications", [])):
                    return 1, "answer in prescription text"
            if any(k in rationale.lower() for k in ["vital", "actual value", "history",
                                                      "patient context", "patient's actual"]):
                if cites_patient_specific_value(answer, patient):
                    return 1, "got specific value right"
                return 0, "needed patient-specific data"
            if "lifestyle" in rationale.lower():
                return 0, "needed lifestyle context"
            if len(answer) > 100:
                return 1, "RAG-expected, reasonable answer"
            return 0, "too generic"
        else:
            return 1, "patient context applied"

    if expected == "RAG_KG":
        if mentions_drug_interaction(answer) and gives_specific_safety_advice(answer):
            return 1, "interaction + concrete advice"
        if gives_specific_safety_advice(answer):
            return 1, "concrete safety advice"
        if is_informative_diplomatic(answer, patient):
            return 1, "diplomatic but informative"
        return 0, "missed safety/interaction"

    return 0, "unknown category"


def score_record(record, patient):
    expected = record["expected_mode_name"]
    rationale = record["rationale"]
    s1, w1 = score_mode(record["mode_1"], patient, expected, rationale)
    s2, w2 = score_mode(record["mode_2"], patient, expected, rationale)
    s3, w3 = score_mode(record["mode_3"], patient, expected, rationale)
    winner = 1 if s1 else (2 if s2 else (3 if s3 else 0))
    return {
        "question_id": record["question_id"],
        "patient_id": record["patient_id"],
        "category": record["category"],
        "expected_mode": record["expected_mode"],
        "expected_mode_name": record["expected_mode_name"],
        "mode_1_correct": s1, "mode_1_rationale": w1,
        "mode_2_correct": s2, "mode_2_rationale": w2,
        "mode_3_correct": s3, "mode_3_rationale": w3,
        "winning_mode": winner,
    }

print("Scoring functions ready")

Scoring functions ready


In [16]:
# Load all responses
responses = []
with open(responses_path) as f:
    for line in f:
        if line.strip():
            responses.append(json.loads(line))

print(f"Loaded {len(responses)} responses")

# Score all
ground_truth = []
for r in tqdm(responses, desc="Scoring"):
    p = patient_by_id[r["patient_id"]]
    gt = score_record(r, p)
    ground_truth.append(gt)

# Save ground truth
gt_path = os.path.join(DRIVE_ROOT, "synthetic_patients", "patient_portal_ground_truth_v2_2000.jsonl")
with open(gt_path, "w") as f:
    for g in ground_truth:
        f.write(json.dumps(g) + "\n")

print(f"\nSaved ground truth to {gt_path}")

# Report
n = len(ground_truth)
print(f"\n{'='*60}")
print(f"Per-mode accuracy (ALL {n} questions):")
print(f"  Mode 1: {sum(g['mode_1_correct'] for g in ground_truth)/n*100:.1f}%")
print(f"  Mode 2: {sum(g['mode_2_correct'] for g in ground_truth)/n*100:.1f}%")
print(f"  Mode 3: {sum(g['mode_3_correct'] for g in ground_truth)/n*100:.1f}%")
print(f"\nWinning mode distribution:")
for m, c in Counter(g['winning_mode'] for g in ground_truth).most_common():
    label = {1:"Mode 1", 2:"Mode 2", 3:"Mode 3", 0:"ALL FAILED"}[m]
    print(f"  {label}: {c} ({c/n*100:.1f}%)")
print(f"{'='*60}")

Loaded 2000 responses


Scoring:   0%|          | 0/2000 [00:00<?, ?it/s]


Saved ground truth to /content/drive/MyDrive/DL Project/synthetic_patients/patient_portal_ground_truth_v2_2000.jsonl

Per-mode accuracy (ALL 2000 questions):
  Mode 1: 82.3%
  Mode 2: 91.3%
  Mode 3: 96.5%

Winning mode distribution:
  Mode 1: 1647 (82.3%)
  Mode 2: 245 (12.2%)
  Mode 3: 70 (3.5%)
  ALL FAILED: 38 (1.9%)


## Cell 8: Build Router Features for All 2000 Questions

In [17]:
# Router feature extraction (from Patient_Portal_Router.ipynb)
CATEGORY_ORDER = [
    "drug_general_info", "drug_mechanism", "drug_interaction", "drug_food_interaction",
    "drug_lifestyle_safety", "drug_side_effects",
    "patient_indication", "patient_timing", "patient_vitals", "patient_lifestyle",
    "patient_side_effects", "patient_monitoring",
]
CATEGORY_TO_IDX = {c: i for i, c in enumerate(CATEGORY_ORDER)}
SAFETY_RELATIONS = {"drug-drug interaction", "synergistic interaction", "side effect", "contraindication"}


def get_retrieval_features(response):
    chunks = response.get("mode_2", {}).get("retrieved_chunks", [])
    scores = [c.get("score", 0.0) for c in chunks]
    top1 = scores[0] if len(scores) >= 1 else 0.0
    top2 = scores[1] if len(scores) >= 2 else 0.0
    gap = top1 - top2
    mean3 = float(np.mean(scores[:3])) if scores else 0.0
    return top1, top2, gap, mean3


def get_kg_features(response):
    n = response.get("mode_3", {}).get("n_kg_triples", 0)
    triples = response.get("mode_3", {}).get("kg_triples", [])
    has_safety = 0
    for t in triples:
        if any(rel in t.lower() for rel in SAFETY_RELATIONS):
            has_safety = 1
            break
    return n, has_safety


def get_patient_features(patient):
    n_meds = len(patient.get("medications", []))
    n_cond = len(patient.get("active_conditions", []))
    has_allerg = 1 if len(patient.get("allergies", [])) > 0 else 0
    return n_meds, n_cond, has_allerg


def count_drugs_in_question(question_text, patient):
    q = question_text.lower()
    n = 0
    for med in patient.get("medications", []):
        first_word = med["drug"].split()[0].lower()
        if first_word in q:
            n += 1
    return n


def build_features(question_record, patient, response):
    """Return a 23-dim numeric feature vector."""
    # 12 one-hot for question type
    type_oh = [0] * len(CATEGORY_ORDER)
    cat = question_record.get("category")
    if cat in CATEGORY_TO_IDX:
        type_oh[CATEGORY_TO_IDX[cat]] = 1

    # 4 retrieval features
    top1, top2, gap, mean3 = get_retrieval_features(response)

    # 2 KG features
    n_kg, has_safety = get_kg_features(response)

    # 3 patient features
    n_meds, n_cond, has_allerg = get_patient_features(patient)

    # 2 question features
    q_len = len(question_record["question"].split())
    n_drugs_q = count_drugs_in_question(question_record["question"], patient)

    return type_oh + [top1, top2, gap, mean3, n_kg, has_safety,
                       n_meds, n_cond, has_allerg,
                       q_len, n_drugs_q]


FEATURE_NAMES = (
    [f"qtype_{c}" for c in CATEGORY_ORDER] +
    ["top1_chunk_score", "top2_chunk_score", "gap_top1_top2_chunk", "mean_top3_chunk_score"] +
    ["n_kg_triples", "has_safety_relation_in_kg"] +
    ["n_active_medications", "n_active_conditions", "has_documented_allergies"] +
    ["question_length_words", "n_drugs_mentioned"]
)
print(f"Total features: {len(FEATURE_NAMES)}")
assert len(FEATURE_NAMES) == 23, "Feature count mismatch!"

Total features: 23


In [18]:
# Build feature matrix for all 2000 questions
resp_by_qid = {r["question_id"]: r for r in responses}

X = []
y = []
question_ids_in_order = []

for q in tqdm(all_questions, desc="Building feature matrix"):
    pid = q["patient_id"]
    patient = patient_by_id[pid]
    response = resp_by_qid.get(q["question_id"])
    if response is None:
        continue
    feat = build_features(q, patient, response)
    X.append(feat)
    y.append(q["expected_mode"] - 1)  # 0-indexed for softmax
    question_ids_in_order.append(q["question_id"])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)

print(f"Feature matrix: {X.shape}")
print(f"Labels: {y.shape}, classes: {np.unique(y, return_counts=True)}")

# Save features to npz
features_path = os.path.join(DRIVE_ROOT, "patient_router_v2", "features_v2.npz")
np.savez(features_path, X=X, y=y, question_ids=np.array(question_ids_in_order, dtype=object))
print(f"Saved features: {features_path}")

Building feature matrix:   0%|          | 0/2000 [00:00<?, ?it/s]

Feature matrix: (2000, 23)
Labels: (2000,), classes: (array([0, 1, 2]), array([536, 867, 597]))
Saved features: /content/drive/MyDrive/DL Project/patient_router_v2/features_v2.npz


## Cell 9: Split Train/Test and Train Router

In [19]:
# Split: 1600 train (first indices), 400 test (last indices)
n_train = len(train_questions)
n_test = len(test_questions)

train_mask = np.array([qid in set(q["question_id"] for q in train_questions) for qid in question_ids_in_order])
test_mask = ~train_mask

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution: {np.unique(y_train, return_counts=True)}")
print(f"Test class distribution: {np.unique(y_test, return_counts=True)}")

Train: (1600, 23), Test: (400, 23)
Train class distribution: (array([0, 1, 2]), array([427, 688, 485]))
Test class distribution: (array([0, 1, 2]), array([109, 179, 112]))


In [20]:
# Router MLP (from Patient_Portal_Router.ipynb)
class RouterMLP(nn.Module):
    def __init__(self, input_dim=23, hidden1=32, hidden2=16, n_classes=3, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden2, n_classes),
        )

    def forward(self, x):
        return self.net(x)


def train_one_fold(X_tr, y_tr, X_va, y_va, epochs=300, batch_size=32,
                    lr=1e-3, patience=40):
    model = RouterMLP(input_dim=X_tr.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()

    Xt = torch.tensor(X_tr).to(DEVICE)
    yt = torch.tensor(y_tr).to(DEVICE)
    Xv = torch.tensor(X_va).to(DEVICE)
    yv = torch.tensor(y_va).to(DEVICE)

    best_val_acc = 0.0
    best_state, bad = None, 0
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), batch_size):
            idx = perm[i:i+batch_size]
            logits = model(Xt[idx])
            loss = crit(logits, yt[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            preds = model(Xv).argmax(-1)
            val_acc = (preds == yv).float().mean().item()
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val_acc

print("Router MLP ready")

Router MLP ready


In [21]:
# 5-fold CV on training data
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_accs = []
fold_predictions = np.zeros(len(y_train), dtype=np.int64) - 1

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
    sc = StandardScaler().fit(X_train[tr_idx])
    X_tr = sc.transform(X_train[tr_idx]).astype(np.float32)
    X_va = sc.transform(X_train[va_idx]).astype(np.float32)

    model, val_acc = train_one_fold(X_tr, y_train[tr_idx], X_va, y_train[va_idx])
    fold_accs.append(val_acc)

    model.eval()
    with torch.no_grad():
        fold_predictions[va_idx] = model(torch.tensor(X_va).to(DEVICE)).argmax(-1).cpu().numpy()

    print(f"Fold {fold}: val_acc = {val_acc*100:.1f}%")

print(f"\n5-fold CV: train_acc = {np.mean(fold_accs)*100:.2f}% +/- {np.std(fold_accs)*100:.2f}%")

Fold 0: val_acc = 98.8%
Fold 1: val_acc = 100.0%
Fold 2: val_acc = 99.4%
Fold 3: val_acc = 99.4%
Fold 4: val_acc = 99.1%

5-fold CV: train_acc = 99.31% +/- 0.41%


In [22]:
# Train final model on all train data
final_scaler = StandardScaler().fit(X_train)
X_train_scaled = final_scaler.transform(X_train).astype(np.float32)

final_model = RouterMLP(input_dim=X_train.shape[1]).to(DEVICE)
opt = torch.optim.Adam(final_model.parameters(), lr=1e-3, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()

Xt = torch.tensor(X_train_scaled).to(DEVICE)
yt = torch.tensor(y_train).to(DEVICE)

print(f"Training final model on {len(X_train)} train samples...")
for epoch in tqdm(range(500), desc="Training final model"):
    final_model.train()
    perm = torch.randperm(len(Xt))
    for i in range(0, len(Xt), 32):
        idx = perm[i:i+32]
        logits = final_model(Xt[idx])
        loss = crit(logits, yt[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()

# Save model + scaler
router_dir = os.path.join(DRIVE_ROOT, "patient_router_v2")
os.makedirs(router_dir, exist_ok=True)

router_path = os.path.join(router_dir, "router_mlp_v2.pt")
torch.save(final_model.state_dict(), router_path)
print(f"Saved router: {router_path}")

scaler_path = os.path.join(router_dir, "scaler_v2.pkl")
with open(scaler_path, "wb") as f:
    pickle.dump(final_scaler, f)
print(f"Saved scaler: {scaler_path}")

Training final model on 1600 train samples...


Training final model:   0%|          | 0/500 [00:00<?, ?it/s]

Saved router: /content/drive/MyDrive/DL Project/patient_router_v2/router_mlp_v2.pt
Saved scaler: /content/drive/MyDrive/DL Project/patient_router_v2/scaler_v2.pkl


In [23]:
# Evaluate on TEST set
X_test_scaled = final_scaler.transform(X_test).astype(np.float32)

final_model.eval()
with torch.no_grad():
    logits = final_model(torch.tensor(X_test_scaled).to(DEVICE))
    probs = F.softmax(logits, dim=-1).cpu().numpy()
    preds_test = probs.argmax(-1)

test_acc = (preds_test == y_test).mean()
print(f"\nTest routing accuracy: {test_acc*100:.2f}% ({preds_test.shape[0]} questions)")

# Confusion matrix
cm = confusion_matrix(y_test, preds_test, labels=[0, 1, 2])
print("\nTest Confusion matrix:")
print(f"           Pred M1  Pred M2  Pred M3")
for i, true_label in enumerate(["True M1", "True M2", "True M3"]):
    print(f"  {true_label:<10} {cm[i, 0]:>7} {cm[i, 1]:>8} {cm[i, 2]:>8}")


Test routing accuracy: 96.75% (400 questions)

Test Confusion matrix:
           Pred M1  Pred M2  Pred M3
  True M1         99        3        7
  True M2          1      176        2
  True M3          0        0      112


## Cell 10: Final Evaluation Summary

In [24]:
# Evaluation strategies on test set
# Build per-question evaluation data
test_qids = np.array(question_ids_in_order)[test_mask]
test_gt_by_qid = {g["question_id"]: g for g in ground_truth}
test_q_by_qid = {q["question_id"]: q for q in test_questions}

joined = []
for i, qid in enumerate(test_qids):
    if qid not in test_gt_by_qid or qid not in test_q_by_qid:
        continue
    g = test_gt_by_qid[qid]
    q = test_q_by_qid[qid]
    joined.append({
        "question_id": qid,
        "expected_mode": q["expected_mode"],
        "router_predicted_mode": int(preds_test[i]) + 1,
        "mode_1_correct": g["mode_1_correct"],
        "mode_2_correct": g["mode_2_correct"],
        "mode_3_correct": g["mode_3_correct"],
    })


def evaluate_strategy(joined, picker, name):
    n = len(joined)
    correct = 0
    for row in joined:
        m = picker(row)
        correct += row[f"mode_{m}_correct"]
    return {"name": name, "accuracy": correct / n * 100, "n_correct": correct, "n": n}


def always(m):
    return lambda row: m


def router_picker(row):
    return row["router_predicted_mode"]


def oracle_picker(row):
    for m in [1, 2, 3]:
        if row[f"mode_{m}_correct"]:
            return m
    return 1


strategies = [
    evaluate_strategy(joined, always(1), "Always Mode 1"),
    evaluate_strategy(joined, always(2), "Always Mode 2"),
    evaluate_strategy(joined, always(3), "Always Mode 3"),
    evaluate_strategy(joined, router_picker, "Router (MLP)"),
    evaluate_strategy(joined, oracle_picker, "Oracle"),
]

print("\n" + "="*70)
print(f"{'Strategy':<22} {'Accuracy':>15} {'Correct':>10}")
print("-" * 70)
for s in strategies:
    print(f"{s['name']:<22} {s['accuracy']:>14.2f}% {s['n_correct']:>10}")
print("=" * 70)


Strategy                      Accuracy    Correct
----------------------------------------------------------------------
Always Mode 1                   80.00%        320
Always Mode 2                   93.00%        372
Always Mode 3                   96.25%        385
Router (MLP)                    95.75%        383
Oracle                          98.50%        394


In [25]:
# Final summary
train_gt = [g for q in train_questions for g in ground_truth if g["question_id"] == q["question_id"]]

summary = {
    "n_patients": len(patients),
    "n_train_questions": len(train_questions),
    "n_test_questions": len(test_questions),
    "n_total_questions": len(all_questions),
    "model_version": "v2_overnight",
    "train_metrics": {
        "mode_1_accuracy": round(sum(g["mode_1_correct"] for g in train_gt) / len(train_gt) * 100, 2),
        "mode_2_accuracy": round(sum(g["mode_2_correct"] for g in train_gt) / len(train_gt) * 100, 2),
        "mode_3_accuracy": round(sum(g["mode_3_correct"] for g in train_gt) / len(train_gt) * 100, 2),
        "router_cv_accuracy": round(np.mean(fold_accs) * 100, 2),
    },
    "test_metrics": {
        "mode_1_accuracy": round(sum(g["mode_1_correct"] for g in ground_truth if g["question_id"] in [q["question_id"] for q in test_questions]) / len(test_questions) * 100, 2),
        "mode_2_accuracy": round(sum(g["mode_2_correct"] for g in ground_truth if g["question_id"] in [q["question_id"] for q in test_questions]) / len(test_questions) * 100, 2),
        "mode_3_accuracy": round(sum(g["mode_3_correct"] for g in ground_truth if g["question_id"] in [q["question_id"] for q in test_questions]) / len(test_questions) * 100, 2),
        "router_accuracy": round(test_acc * 100, 2),
        "oracle_accuracy": round(sum(row["mode_1_correct"] or row["mode_2_correct"] or row["mode_3_correct"] for row in joined) / len(joined) * 100, 2),
    },
    "strategies": [{
        "name": s["name"],
        "test_accuracy": round(s["accuracy"], 2),
    } for s in strategies],
}

summary_path = os.path.join(DRIVE_ROOT, "patient_router_v2", "v2_eval_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved summary: {summary_path}")
print("\n" + "="*70)
print("FINAL EVALUATION SUMMARY")
print("=" * 70)
print(json.dumps(summary, indent=2))
print("=" * 70)


Saved summary: /content/drive/MyDrive/DL Project/patient_router_v2/v2_eval_summary.json

FINAL EVALUATION SUMMARY
{
  "n_patients": 400,
  "n_train_questions": 1600,
  "n_test_questions": 400,
  "n_total_questions": 2000,
  "model_version": "v2_overnight",
  "train_metrics": {
    "mode_1_accuracy": 82.94,
    "mode_2_accuracy": 90.94,
    "mode_3_accuracy": 96.56,
    "router_cv_accuracy": 99.31
  },
  "test_metrics": {
    "mode_1_accuracy": 80.0,
    "mode_2_accuracy": 93.0,
    "mode_3_accuracy": 96.25,
    "router_accuracy": 96.75,
    "oracle_accuracy": 98.5
  },
  "strategies": [
    {
      "name": "Always Mode 1",
      "test_accuracy": 80.0
    },
    {
      "name": "Always Mode 2",
      "test_accuracy": 93.0
    },
    {
      "name": "Always Mode 3",
      "test_accuracy": 96.25
    },
    {
      "name": "Router (MLP)",
      "test_accuracy": 95.75
    },
    {
      "name": "Oracle",
      "test_accuracy": 98.5
    }
  ]
}
